# 02 — GenerationConfig: All Sampling Strategies

Every knob on `GenerationConfig` explained and compared:

| Strategy | Key fields |
|----------|------------|
| Greedy | `do_sample=False` |
| Temperature | `temperature`, `do_sample=True` |
| Top-K | `top_k=N` |
| Top-P (nucleus) | `top_p=0.95` |
| Repetition penalty | `repetition_penalty=1.2` |
| KV cache | `use_kv_cache=True` |
| Log-probs | `return_logprobs=True` |

> **Colab users:** Run the setup cell, restart runtime, then continue.


In [1]:
import subprocess, sys, os

def _is_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if _is_colab():
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install',
         'git+https://github.com/silvaxxx1/MyLLM.git'],
        capture_output=True, text=True
    )
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Done! Now go to Runtime → Restart runtime, then continue.')
else:
    root = os.path.abspath(os.path.join(os.getcwd(), '..'))
    has_uv = subprocess.run(['which', 'uv'], capture_output=True).returncode == 0
    cmd = ['uv', 'pip', 'install', '-e', root] if has_uv else [sys.executable, '-m', 'pip', 'install', '-e', root]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('Install FAILED:', r.stderr[-2000:])
    else:
        print('Local editable install ready.')


Local editable install ready.


In [2]:
from myllm import LLM, GenerationConfig

llm = LLM.from_pretrained('gpt2-small')
print(llm)

PROMPT = 'Scientists recently discovered that'

def gen(cfg: GenerationConfig) -> str:
    return llm.generate_text(PROMPT, generation_config=cfg, skip_prompt=True)['text']

print('Ready.')


2026-05-09 20:09:07.342385: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


📋 Using custom config
📊 Estimated memory: 0.71 GB
   Parameters: 0.57 GB
   Activations: 0.14 GB
⬇️  Downloading gpt2-small...
✅ File ./models/model-gpt2-small.safetensors already exists and is valid.
📂 Loading weights from disk...
📦 Loading weights from ./models/model-gpt2-small.safetensors to cpu...
✅ Loaded 160 tensors
🏗️  Initializing model architecture...
🔧 Converting model to torch.float32...
🔄 Mapping weights using gpt2_mapper...
Loading gpt2-small weights -

Loading GPT-2 blocks:   0%|          | 0/12 [00:00<?, ?block/s]

Loading gpt2-small weights /

Loading GPT-2 blocks:   8%|▊         | 1/12 [00:01<00:17,  1.59s/block]

...

Loading gpt2-small weights |

Loading GPT-2 blocks:  83%|████████▎ | 10/12 [00:07<00:00,  2.52block/s]

Loading gpt2-small weights /

Loading GPT-2 blocks: 100%|██████████| 12/12 [00:07<00:00,  1.58block/s]


✅ Successfully loaded gpt2-small!
🎯 Model ready on cuda for inference!
LLM(model='gpt2-small', params=163.0M, device='cuda', dtype=torch.float32)
Ready.


## 1. Greedy decoding

Always picks the highest-probability token. Deterministic — same output every run.

In [3]:
cfg = GenerationConfig(
    max_length=50,
    do_sample=False,
    use_kv_cache=True,
    use_optimized_sampler=False,
)
print('Greedy:')
print(gen(cfg))

Greedy:
 the brain of a man who had been diagnosed with Alzheimer's disease had a different structure than that of a woman who had not.

The researchers found that the brain of a man who had been diagnosed with Alzheimer's disease had a different structure than


## 2. Temperature

- `< 1.0` → sharper → more focused
- `> 1.0` → flatter → more random
- `= 1.0` → unmodified model distribution

In [4]:
for temp in [0.3, 0.7, 1.0, 1.5]:
    cfg = GenerationConfig(
        max_length=40, temperature=temp, do_sample=True,
        use_kv_cache=True,
    )
    print(f'temperature={temp}:')
    print(' ', gen(cfg))
    print()


temperature=0.3:
   the same protein that makes up the protein that makes up the protein that makes up the protein that makes up the protein that makes up the protein that makes up the protein that makes up the protein that makes

temperature=0.7:
   a protein called Wnt1 that tells bacteria to fight off pathogens without getting sick occurs after they become exposed to viruses. The Wnt1 protein is present in about 18 percent of viruses and about 20

temperature=1.0:
   hoarders steal an unknown form of gold's body to make a point, but just waiting for your suspicions. Learn the secrets of low-cost luxury journeys…<|endoftext|>MOSCOW (Reuters)

temperature=1.5:
   changes from nudger association sernodeistan mobilization impair response to begin collision thrusters training ," because forwards approaching group cockroach serumnWhy humans are curing a bird with mud removal: part Icosystemcat



## 3. Top-K sampling

Keep only the top K tokens at each step and renormalise.

In [5]:
for k in [5, 20, 50, 200]:
    cfg = GenerationConfig(
        max_length=40, temperature=1.0, do_sample=True,
        top_k=k, use_kv_cache=True, use_optimized_sampler=True,
    )
    print(f'top_k={k}:')
    print(' ', gen(cfg))
    print()


top_k=5:
   a single cell of the human brain is composed of about 1 percent of the cells of the animal, and the other 10 percent are the same. This is because the cells of the animal have different functions

top_k=20:
   these species of fungi live in a very hot and humid climate, where it is more difficult to grow their plants.

According to the team, they used a method to find out whether that temperature

top_k=50:
   all life on Earth is composed of many, but are all at the same stage of formation. It seemed that a new type of life existed not in our planet but in our solar system. Since its

top_k=200:
   there is a strong need in biology for an ocean-forming system that serves as a buffer to harm from weather and extreme events like typhoons.

The scientists know that "no species can tolerate



## 4. Top-P (nucleus) sampling

Keep the smallest set of tokens whose cumulative probability ≥ p.

In [6]:
for p in [0.5, 0.8, 0.95, 1.0]:
    cfg = GenerationConfig(
        max_length=40, temperature=1.0, do_sample=True,
        top_p=p, use_kv_cache=True, use_optimized_sampler=True,
    )
    print(f'top_p={p}:')
    print(' ', gen(cfg))
    print()


top_p=0.5:
   white-matter cells have been used to study cancer cells. The researchers have been able to find out more about the mechanisms of cancer cell growth and metastasis.

"We've shown that we

top_p=0.8:
   the brown bumblebee has developed a tardigrade with new characteristics that make it a close cousin to the human "fingerspine" beetle, which has adapted to its environment. This new

top_p=0.95:
   there's an active underground core from who doesn't think."

Wherever you look and if you can show you an image that took pictures at almost 100x magnification – and deserves your content –

top_p=1.0:
   the waves of genetic drift he notes, common to forensic paths of the historical era, focus location on a fleeting or even perpetual source of genetic clockwork resulting in the construction of new traces of Han return



## 5. Top-K + Top-P combined (recommended default)

In [7]:
cfg = GenerationConfig(
    max_length=80, temperature=0.8, do_sample=True,
    top_k=50, top_p=0.95,
    use_kv_cache=True, use_optimized_sampler=True,
)
print('Top-K=50 + Top-P=0.95 + temp=0.8:')
print(gen(cfg))


Top-K=50 + Top-P=0.95 + temp=0.8:
 a large number of genes involved in serotonin (S)-reuptake inhibitors (SSRIs) are directly involved in serotonin release. These include the genes involved in the dopamine transporter (DAT) and the receptor involved in serotonin release.

The authors of the study, Dr. Nasser J. Abouy and his team at The Johns Hopkins School of Medicine in Baltimore, MD,


## 6. Repetition penalty

Divides the logit of any already-seen token by `repetition_penalty`. Values > 1.0 discourage repeating.

In [8]:
for penalty in [1.0, 1.2, 1.5, 2.0]:
    cfg = GenerationConfig(
        max_length=60, temperature=0.9, do_sample=True,
        top_k=50, repetition_penalty=penalty,
        use_kv_cache=True, use_optimized_sampler=True,
    )
    print(f'repetition_penalty={penalty}:')
    print(' ', gen(cfg))
    print()


repetition_penalty=1.0:
   "numerous species of birds" may have evolved as "mammals," one of the more surprising discoveries of its kind. Their eggs lay in a large brood of many species of birds, but their numbers are limited to the few that are known to exist. There are three major types of m

repetition_penalty=1.2:
   that recently discovered recently discovered that recently discovered recently discovered that recently discovered that recently discovered that recently discovered that recently discovered that recently discovered that recently discovered that recently discovered that recently discovered that recently discovered that recently discovered

Scientists recently discovered three recently discovered recently noted that recently discovered that recently discovered they recently discovered that

repetition_penalty=1.5:
   recently discovered discovered that recently discovered that discovered that discovered that discovered that discovered that discovered that discovered that

## 7. KV cache speed comparison

In [9]:
import time, torch

input_ids = tokenizer.encode(PROMPT, return_tensors='pt').to(device)

for use_kv in [True, False]:
    cfg = GenerationConfig(
        max_length=50, temperature=1.0, do_sample=True,
        top_k=50, use_kv_cache=use_kv, use_optimized_sampler=True,
    )
    with torch.no_grad(): llm.generate(input_ids, cfg)  # warmup

    t0 = time.perf_counter()
    with torch.no_grad(): out = llm.generate(input_ids, cfg)
    elapsed = time.perf_counter() - t0
    n = out['tokens'].numel()
    print(f'use_kv_cache={str(use_kv):<5}  {n} tokens  {elapsed:.3f}s  {n/elapsed:.1f} tok/s')


NameError: name 'tokenizer' is not defined

## 8. Log-probabilities

In [ ]:
import torch

cfg = GenerationConfig(
    max_length=15, temperature=0.8, do_sample=True,
    top_k=50, use_kv_cache=True, return_logprobs=True,
    use_optimized_sampler=True,
)

input_ids = tokenizer.encode(PROMPT, return_tensors='pt').to(device)
output    = llm.generate(input_ids, cfg)

prompt_len       = input_ids.shape[1]
generated_tokens = output['tokens'][0][prompt_len:]
logprobs         = output['logprobs'][0]

print(f'{"Token":<20}  Log-prob   Prob')
print('-' * 42)
for tok_id, lp in zip(generated_tokens.tolist(), logprobs.tolist()):
    word = tokenizer.decode([tok_id])
    prob = torch.exp(torch.tensor(lp)).item()
    print(f'{repr(word):<20}  {lp:>8.3f}   {prob:.4f}')
